In [21]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

Шаг 3.2 — load raw dataset

In [22]:
DATA_PATH = Path("../data/raw/WineQT.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


In [23]:
df.shape

(1143, 13)

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB


Шаг 3.3 — create X and y

In [25]:
TARGET = "quality"
ID_COL = "Id"

y = df[TARGET]

X = df.drop(columns=[TARGET, ID_COL])

X.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4


In [26]:
X.shape, y.shape

((1143, 11), (1143,))

In [27]:
X.columns

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol'],
      dtype='str')

In [28]:
y.head()

0    5
1    5
2    5
3    6
4    5
Name: quality, dtype: int64

Шаг 3.4 — train/test split

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [30]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((914, 11), (229, 11), (914,), (229,))

Шаг 3.5 — check target summary in train/test

Так как это regression, нам важны не только counts, но и summary.

In [31]:
target_summary = pd.DataFrame({
    "train": y_train.describe(),
    "test": y_test.describe(),
})

target_summary

,train,test
count,914.000000,229.000000
mean,5.657549,5.655022
std,0.806472,0.804992
min,3.000000,3.000000
25%,5.000000,5.000000
50%,6.000000,6.000000
75%,6.000000,6.000000
max,8.000000,8.000000


In [32]:
target_summary_value_counts = pd.DataFrame({
    "train": y_train.value_counts(),
    "test": y_test.value_counts(),
})

target_summary_value_counts

,train,test
quality,,
5,386,97
6,370,92
7,114,29
4,26,7
8,13,3
3,5,1


Распределение классов quality в train/test тоже полезно, потому что target ordinal/discrete:

In [33]:
train_quality_distribution = (
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .rename("train_proportion")
)

test_quality_distribution = (
    y_test
    .value_counts(normalize=True)
    .sort_index()
    .rename("test_proportion")
)

quality_distribution_split = pd.concat(
    [train_quality_distribution, test_quality_distribution],
    axis=1,
)

quality_distribution_split

,train_proportion,test_proportion
quality,,
3,0.005470,0.004367
4,0.028446,0.030568
5,0.422319,0.423581
6,0.404814,0.401747
7,0.124726,0.126638
8,0.014223,0.013100


The test set was created once and will remain untouched until final evaluation.
All baseline model comparison in this stage is performed using cross-validation on the training set only.

Шаг 3.6 — define CV

In [34]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

Шаг 3.7 — define metrics

In [35]:
scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

Шаг 3.8 — define baseline models

Здесь важный момент: для Ridge используем scaling внутри Pipeline. Для LinearRegression scaling не обязателен для качества OLS, но можно тоже положить в Pipeline для единообразия. Однако для совсем простого baseline оставим так:

Почему DecisionTreeRegressor(max_depth=3): это не tuning, а ограниченная простая baseline-модель, чтобы не получить переобученное дерево уже на первом шаге.

In [36]:
models = {
    "dummy_mean": DummyRegressor(strategy="mean"),
    "linear_regression": LinearRegression(),
    "ridge_scaled": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ]),
    "decision_tree_depth_3": DecisionTreeRegressor(
        max_depth=3,
        random_state=42,
    ),
}

Шаг 3.9 — cross-validation function

In [ ]:
def evaluate_model_cv(model, X_train, y_train, cv, scoring):
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
    )
    
    result = {
        "mae_mean": -cv_results["test_mae"].mean(),
        "mae_std": cv_results["test_mae"].std(),
        "rmse_mean": -cv_results["test_rmse"].mean(),
        "rmse_std": cv_results["test_rmse"].std(),
        "r2_mean": cv_results["test_r2"].mean(),
        "r2_std": cv_results["test_r2"].std(),
    }
    
    return result

Шаг 3.10 — evaluate all models

In [ ]:
results = []

for model_name, model in models.items():
    model_result = evaluate_model_cv(
        model=model,
        X_train=X_train,
        y_train=y_train,
        cv=cv,
        scoring=scoring,
    )
    
    model_result["model"] = model_name
    results.append(model_result)

baseline_results = (
    pd.DataFrame(results)
    .set_index("model")
    .sort_values("rmse_mean")
)

baseline_results

,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
model,,,,,,
ridge_scaled,0.513299,0.022894,0.659862,0.036905,0.346446,0.042546
linear_regression,0.513352,0.022988,0.659970,0.037029,0.346239,0.042651
decision_tree_depth_3,0.547243,0.018165,0.693632,0.023701,0.277215,0.030562
dummy_mean,0.689707,0.029457,0.819602,0.043050,-0.006710,0.007733


Шаг 3.11 — readable rounded table

In [ ]:
baseline_results.round(4)

,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
model,,,,,,
ridge_scaled,0.5133,0.0229,0.6599,0.0369,0.3464,0.0425
linear_regression,0.5134,0.0230,0.6600,0.0370,0.3462,0.0427
decision_tree_depth_3,0.5472,0.0182,0.6936,0.0237,0.2772,0.0306
dummy_mean,0.6897,0.0295,0.8196,0.0431,-0.0067,0.0077


Шаг 3.12 — baseline conclusions markdown

## Baseline regression conclusions

### Setup

- Task type: regression.
- Target: `quality`.
- Excluded column: `Id`.
- Feature matrix contains 11 physicochemical features.
- Train/test split was created with:
  - `test_size=0.2`;
  - `random_state=42`.
- The test set was not used for model evaluation in this stage.
- All model comparison was performed using 5-fold cross-validation on `X_train` only.

### Models evaluated

- `DummyRegressor(strategy="mean")`;
- `LinearRegression`;
- `Ridge(alpha=1.0)` with `StandardScaler` inside Pipeline;
- `DecisionTreeRegressor(max_depth=3)`.

### Metrics

- MAE;
- RMSE;
- R².

### Main findings

Write here:
- which model performed best by RMSE;
- whether simple models improved over DummyRegressor;
- whether R² is meaningfully above zero;
- whether the baseline performance looks modest or strong;
- why this is only a starting point, not final model selection.

### Leakage control

- No preprocessing was fitted on the full dataset.
- Scaling for Ridge was placed inside Pipeline.
- Cross-validation was performed only on the training set.
- The test set remains untouched until final evaluation.

## Baseline regression conclusions

### Setup

- Task type: regression.
- Target: `quality`.
- Excluded column: `Id`.
- Feature matrix contains 11 physicochemical features.
- Train/test split was created with:
  - `test_size=0.2`;
  - `random_state=42`;
  - `stratify=y`.

Although the task is framed as regression, the target is a discrete ordinal score. Therefore, the holdout split was stratified by `quality` to preserve the target distribution across train and test.

The test set was created but not used for model evaluation in this stage. All baseline model comparison was performed using 5-fold cross-validation on `X_train` only.

### Models evaluated

- `DummyRegressor(strategy="mean")`;
- `LinearRegression`;
- `Ridge(alpha=1.0)` with `StandardScaler` inside Pipeline;
- `DecisionTreeRegressor(max_depth=3)`.

### Metrics

- MAE;
- RMSE;
- R².

### Main findings

- The best baseline models by RMSE were `ridge_scaled` and `linear_regression`.
- Their performance was almost identical:
  - `ridge_scaled`: MAE ≈ 0.5133, RMSE ≈ 0.6599, R² ≈ 0.3464;
  - `linear_regression`: MAE ≈ 0.5134, RMSE ≈ 0.6600, R² ≈ 0.3462.
- Both linear models clearly improved over the dummy baseline:
  - `dummy_mean`: MAE ≈ 0.6897, RMSE ≈ 0.8196, R² ≈ -0.0067.
- The limited decision tree also improved over the dummy baseline, but performed worse than the linear models:
  - `decision_tree_depth_3`: MAE ≈ 0.5472, RMSE ≈ 0.6936, R² ≈ 0.2772.
- MAE around 0.51 means that the best baseline model is wrong by about half a quality point on average.
- R² around 0.35 suggests that the baseline captures some useful signal, but there is still substantial unexplained variation.
- This stage establishes honest baseline performance; it is not final model selection and does not include hyperparameter tuning.

### Leakage control

- `Id` was excluded from the feature matrix.
- No preprocessing was fitted on the full dataset.
- Scaling for Ridge was placed inside a `Pipeline`.
- Cross-validation was performed only on the training set.
- The holdout test set remains untouched until final evaluation.
- No outliers were removed.
- No hyperparameter tuning was performed.